# CSE 472 Assignment 2: Decision Trees, Random Forests, and Extra Trees

**Student ID:** 2005095

This notebook implements tree-based classifiers from scratch and compares them with scikit-learn implementations.

## Objectives
- Understand the working principles of decision trees
- Implement ensemble techniques such as Random Forests and Extra Trees
- Analyze bias–variance tradeoffs empirically
- Compare custom implementations with industrial-grade libraries

In [ ]:
import numpy as np
from collections import Counter
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier as SklearnDT
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.ensemble import ExtraTreesClassifier as SklearnET
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Libraries imported successfully!")

## 2. Decision Tree Implementation

### Decision Tree Node Class
A node in the decision tree that stores split information or leaf predictions.

In [ ]:
class DecisionTreeNode:
    """A node in the decision tree."""
    
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, 
                 value=None, is_leaf=False, class_distribution=None):
        self.feature_index = feature_index  # Index of feature to split on
        self.threshold = threshold          # Threshold value for split
        self.left = left                    # Left child node
        self.right = right                  # Right child node
        self.value = value                  # Class prediction (for leaf nodes)
        self.is_leaf = is_leaf              # Whether this is a leaf node
        self.class_distribution = class_distribution  # Class probabilities in leaf

### Decision Tree Classifier

Custom implementation using Gini impurity for splitting criterion. Supports configurable hyperparameters:
- `max_depth`: Maximum depth of the tree
- `min_samples_split`: Minimum samples required to split a node
- `max_features`: Number of features to consider at each split
- `splitter`: 'best' for optimal split, 'random' for Extra Trees style

In [ ]:
class DecisionTreeClassifier:
    """
    Decision Tree Classifier implemented from scratch.
    
    Uses Gini impurity for splitting criterion and supports configurable
    hyperparameters for tree depth, minimum samples, and feature selection.
    """
    
    def __init__(self, max_depth=None, min_samples_split=2, max_features=None, 
                 random_state=None, splitter='best'):
        """
        Initialize the Decision Tree Classifier.
        
        Parameters:
        -----------
        max_depth : int or None
            Maximum depth of the tree. None means unlimited.
        min_samples_split : int
            Minimum number of samples required to split a node.
        max_features : int, float, str, or None
            Number of features to consider for each split.
            - int: exact number of features
            - float: fraction of features
            - 'sqrt': sqrt(n_features)
            - 'log2': log2(n_features)
            - None: all features
        random_state : int or None
            Random seed for reproducibility.
        splitter : str
            'best' for best split, 'random' for random split (used by Extra Trees)
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.splitter = splitter
        self.root = None
        self.n_classes_ = None
        self.n_features_ = None
        self.classes_ = None
        self.rng = np.random.RandomState(random_state)
    
    def _gini_impurity(self, y):
        """Calculate Gini impurity for a set of labels."""
        if len(y) == 0:
            return 0
        counts = np.bincount(y, minlength=self.n_classes_)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _information_gain(self, y, left_indices, right_indices, parent_gini):
        """Calculate information gain from a split."""
        if len(left_indices) == 0 or len(right_indices) == 0:
            return 0
        
        n = len(y)
        n_left, n_right = len(left_indices), len(right_indices)
        
        left_gini = self._gini_impurity(y[left_indices])
        right_gini = self._gini_impurity(y[right_indices])
        
        child_gini = (n_left / n) * left_gini + (n_right / n) * right_gini
        return parent_gini - child_gini
    
    def _get_n_features_to_consider(self, n_features):
        """Determine number of features to consider for splitting."""
        if self.max_features is None:
            return n_features
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        elif isinstance(self.max_features, float):
            return max(1, int(self.max_features * n_features))
        elif self.max_features == 'sqrt':
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(n_features)))
        else:
            return n_features
    
    def _find_best_split(self, X, y):
        """Find the best feature and threshold for splitting."""
        n_samples, n_features = X.shape
        
        if n_samples < self.min_samples_split:
            return None, None
        
        # Select features to consider
        n_features_to_consider = self._get_n_features_to_consider(n_features)
        feature_indices = self.rng.choice(n_features, n_features_to_consider, replace=False)
        
        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        
        # Calculate parent Gini once
        parent_gini = self._gini_impurity(y)
        
        for feature_idx in feature_indices:
            feature_values = X[:, feature_idx]
            
            if self.splitter == 'random':
                # Extra Trees style: random threshold
                min_val, max_val = feature_values.min(), feature_values.max()
                if min_val == max_val:
                    continue
                threshold = self.rng.uniform(min_val, max_val)
                thresholds = [threshold]
            else:
                # Best split: try all unique thresholds
                unique_values = np.unique(feature_values)
                if len(unique_values) == 1:
                    continue
                thresholds = (unique_values[:-1] + unique_values[1:]) / 2
            
            for threshold in thresholds:
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]
                
                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue
                
                gain = self._information_gain(y, left_indices, right_indices, parent_gini)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree."""
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # Stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_classes == 1 or \
           n_samples < self.min_samples_split:
            # Create leaf node with class distribution
            leaf_value = Counter(y).most_common(1)[0][0]
            class_counts = np.bincount(y, minlength=self.n_classes_)
            class_distribution = class_counts / len(y)
            return DecisionTreeNode(value=leaf_value, is_leaf=True, class_distribution=class_distribution)
        
        # Find best split
        best_feature, best_threshold = self._find_best_split(X, y)
        
        if best_feature is None:
            # No valid split found, create leaf with class distribution
            leaf_value = Counter(y).most_common(1)[0][0]
            class_counts = np.bincount(y, minlength=self.n_classes_)
            class_distribution = class_counts / len(y)
            return DecisionTreeNode(value=leaf_value, is_leaf=True, class_distribution=class_distribution)
        
        # Split data
        left_indices = np.where(X[:, best_feature] <= best_threshold)[0]
        right_indices = np.where(X[:, best_feature] > best_threshold)[0]
        
        # Build child nodes
        left_child = self._build_tree(X[left_indices], y[left_indices], depth + 1)
        right_child = self._build_tree(X[right_indices], y[right_indices], depth + 1)
        
        return DecisionTreeNode(
            feature_index=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child
        )
    
    def fit(self, X, y):
        """Fit the decision tree classifier."""
        X = np.array(X)
        y = np.array(y)
        
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.n_features_ = X.shape[1]
        
        # Map labels to 0, 1, 2, ...
        self.label_map_ = {label: idx for idx, label in enumerate(self.classes_)}
        y_mapped = np.array([self.label_map_[label] for label in y])
        
        self.root = self._build_tree(X, y_mapped)
        return self
    
    def _predict_sample(self, x, node):
        """Predict class for a single sample."""
        if node.is_leaf:
            return node.value
        
        if x[node.feature_index] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict class labels for samples in X."""
        X = np.array(X)
        predictions = np.array([self._predict_sample(x, self.root) for x in X])
        return np.array([self.classes_[pred] for pred in predictions])
    
    def _predict_proba_sample(self, x, node):
        """Get class probabilities for a single sample by traversing to leaf."""
        if node.is_leaf:
            return node.class_distribution
        
        if x[node.feature_index] <= node.threshold:
            return self._predict_proba_sample(x, node.left)
        else:
            return self._predict_proba_sample(x, node.right)
    
    def predict_proba(self, X):
        """Predict class probabilities for samples in X."""
        X = np.array(X)
        return np.array([self._predict_proba_sample(x, self.root) for x in X])

print("DecisionTreeClassifier class defined!")

## 3. Random Forest Implementation

Random Forest uses:
1. **Bootstrap sampling** - Each tree is trained on a random sample with replacement
2. **Feature randomization** - Only a subset of features is considered at each split
3. **Majority voting** - Final prediction is based on votes from all trees

In [ ]:
class RandomForestClassifier:
    """
    Random Forest Classifier implemented from scratch.
    
    Uses bootstrap sampling and feature randomization to build an ensemble
    of decision trees.
    """
    
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2,
                 max_features='sqrt', random_state=None):
        """
        Initialize the Random Forest Classifier.
        
        Parameters:
        -----------
        n_estimators : int
            Number of trees in the forest.
        max_depth : int or None
            Maximum depth of each tree.
        min_samples_split : int
            Minimum samples required to split a node.
        max_features : int, float, str, or None
            Number of features to consider at each split.
        random_state : int or None
            Random seed for reproducibility.
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.classes_ = None
        self.n_classes_ = None
        self.rng = np.random.RandomState(random_state)
    
    def _bootstrap_sample(self, X, y):
        """Create a bootstrap sample of the data."""
        n_samples = X.shape[0]
        indices = self.rng.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]
    
    def fit(self, X, y):
        """Fit the random forest classifier."""
        X = np.array(X)
        y = np.array(y)
        
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.trees = []
        
        for i in range(self.n_estimators):
            # Bootstrap sample
            X_boot, y_boot = self._bootstrap_sample(X, y)
            
            # Create and train tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                random_state=self.rng.randint(0, 10000),
                splitter='best'
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
        
        return self
    
    def predict(self, X):
        """Predict class labels using majority voting."""
        X = np.array(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])
        # Majority voting
        majority_votes = []
        for i in range(X.shape[0]):
            votes = predictions[:, i]
            majority_votes.append(Counter(votes).most_common(1)[0][0])
        return np.array(majority_votes)
    
    def predict_proba(self, X):
        """Predict class probabilities by averaging tree predictions."""
        X = np.array(X)
        all_proba = np.array([tree.predict_proba(X) for tree in self.trees])
        return np.mean(all_proba, axis=0)

print("RandomForestClassifier class defined!")

## 4. Extra Trees Implementation

Unlike Random Forest, Extra Trees (Extremely Randomized Trees):
1. **Uses entire training set** - No bootstrap sampling
2. **Random threshold selection** - Instead of finding the optimal split, it randomly selects a threshold for each feature

This additional randomness often leads to faster training and can reduce variance further.

In [ ]:
class ExtraTreesClassifier:
    """
    Extremely Randomized Trees (Extra Trees) Classifier implemented from scratch.
    
    Unlike Random Forest, Extra Trees:
    1. Uses the entire training set (no bootstrap)
    2. Selects random thresholds for splitting
    """
    
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2,
                 max_features='sqrt', random_state=None):
        """
        Initialize the Extra Trees Classifier.
        
        Parameters:
        -----------
        n_estimators : int
            Number of trees in the ensemble.
        max_depth : int or None
            Maximum depth of each tree.
        min_samples_split : int
            Minimum samples required to split a node.
        max_features : int, float, str, or None
            Number of features to consider at each split.
        random_state : int or None
            Random seed for reproducibility.
        """
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.classes_ = None
        self.n_classes_ = None
        self.rng = np.random.RandomState(random_state)
    
    def fit(self, X, y):
        """Fit the Extra Trees classifier."""
        X = np.array(X)
        y = np.array(y)
        
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.trees = []
        
        for i in range(self.n_estimators):
            # Extra Trees uses entire dataset (no bootstrap)
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                random_state=self.rng.randint(0, 10000),
                splitter='random'  # Key difference: random splits
            )
            tree.fit(X, y)
            self.trees.append(tree)
        
        return self
    
    def predict(self, X):
        """Predict class labels using majority voting."""
        X = np.array(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])
        majority_votes = []
        for i in range(X.shape[0]):
            votes = predictions[:, i]
            majority_votes.append(Counter(votes).most_common(1)[0][0])
        return np.array(majority_votes)
    
    def predict_proba(self, X):
        """Predict class probabilities by averaging tree predictions."""
        X = np.array(X)
        all_proba = np.array([tree.predict_proba(X) for tree in self.trees])
        return np.mean(all_proba, axis=0)

print("ExtraTreesClassifier class defined!")

## 5. Evaluation Functions

Define helper functions to calculate metrics and evaluate models.

In [ ]:
def calculate_metrics(y_true, y_pred, y_proba, classes):
    """
    Calculate evaluation metrics for classification.
    
    Returns:
    --------
    dict : Dictionary containing accuracy, f1_score (macro), and auroc.
    """
    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    
    # Calculate AUROC
    if len(classes) == 2:
        auroc = roc_auc_score(y_true, y_proba[:, 1])
    else:
        y_true_bin = label_binarize(y_true, classes=classes)
        try:
            auroc = roc_auc_score(y_true_bin, y_proba, average='weighted', multi_class='ovr')
        except ValueError:
            auroc = np.nan
    
    return {
        'Accuracy': accuracy,
        'F1-Macro': f1_macro,
        'F1-Weighted': f1_weighted,
        'AUROC': auroc
    }


def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train and evaluate a model."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    classes = np.unique(y_train)
    return calculate_metrics(y_test, y_pred, y_proba, classes)


def run_experiment(dataset_name, X, y, random_state=RANDOM_SEED, n_folds=5):
    """
    Run complete experiment on a dataset using k-fold cross-validation.
    """
    X = np.array(X)
    y = np.array(y)
    
    def get_models(seed):
        return {
            'Custom Decision Tree': DecisionTreeClassifier(
                max_depth=10, min_samples_split=2, random_state=seed
            ),
            'Custom Random Forest': RandomForestClassifier(
                n_estimators=100, max_depth=10, min_samples_split=2,
                max_features='sqrt', random_state=seed
            ),
            'Custom Extra Trees': ExtraTreesClassifier(
                n_estimators=100, max_depth=10, min_samples_split=2,
                max_features='sqrt', random_state=seed
            ),
            'Sklearn Decision Tree': SklearnDT(
                max_depth=10, min_samples_split=2, random_state=seed
            ),
            'Sklearn Random Forest': SklearnRF(
                n_estimators=100, max_depth=10, min_samples_split=2,
                max_features='sqrt', random_state=seed
            ),
            'Sklearn Extra Trees': SklearnET(
                n_estimators=100, max_depth=10, min_samples_split=2,
                max_features='sqrt', random_state=seed
            ),
        }
    
    model_names = list(get_models(0).keys())
    fold_results = {name: {'Accuracy': [], 'F1-Macro': [], 'F1-Weighted': [], 'AUROC': []} 
                    for name in model_names}
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold_idx + 1}/{n_folds}...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        models = get_models(random_state + fold_idx)
        
        for name, model in models.items():
            metrics = evaluate_model(model, X_train, X_test, y_train, y_test, name)
            for metric_name, metric_value in metrics.items():
                fold_results[name][metric_name].append(metric_value)
    
    # Calculate mean and std
    results = {}
    for name in model_names:
        results[name] = {}
        for metric_name in ['Accuracy', 'F1-Macro', 'F1-Weighted', 'AUROC']:
            values = fold_results[name][metric_name]
            valid_values = [v for v in values if not np.isnan(v)]
            if valid_values:
                results[name][metric_name] = np.mean(valid_values)
                results[name][f'{metric_name}_std'] = np.std(valid_values)
            else:
                results[name][metric_name] = np.nan
                results[name][f'{metric_name}_std'] = np.nan
    
    return results

print("Evaluation functions defined!")

## 6. Load Datasets

In [ ]:
# Load datasets
iris = load_iris()
wine = load_wine()

datasets = {
    'Iris': (iris.data, iris.target),
    'Wine': (wine.data, wine.target)
}

print("Datasets loaded:")
for name, (X, y) in datasets.items():
    print(f"  {name}: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

## 7. Run Experiments on Iris Dataset

In [ ]:
print("="*80)
print("Running 5-Fold Cross-Validation on Iris Dataset")
print("="*80)

X_iris, y_iris = datasets['Iris']
iris_results = run_experiment('Iris', X_iris, y_iris)

print("\nExperiment complete!")

### Iris Dataset Results Table

In [ ]:
import pandas as pd

# Create results dataframe for Iris
iris_table_data = []
for model_name, metrics in iris_results.items():
    iris_table_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['Accuracy']:.4f}",
        'F1-Macro': f"{metrics['F1-Macro']:.4f}",
        'AUROC': f"{metrics['AUROC']:.4f}" if not np.isnan(metrics['AUROC']) else "N/A"
    })

iris_df = pd.DataFrame(iris_table_data)
print("="*100)
print("IRIS DATASET RESULTS (5-Fold Cross-Validation)")
print("="*100)
display(iris_df)

## 8. Run Experiments on Wine Dataset

In [ ]:
print("="*80)
print("Running 5-Fold Cross-Validation on Wine Dataset")
print("="*80)

X_wine, y_wine = datasets['Wine']
wine_results = run_experiment('Wine', X_wine, y_wine)

print("\nExperiment complete!")

### Wine Dataset Results Table

In [ ]:
# Create results dataframe for Wine
wine_table_data = []
for model_name, metrics in wine_results.items():
    wine_table_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['Accuracy']:.4f}",
        'F1-Macro': f"{metrics['F1-Macro']:.4f}",
        'AUROC': f"{metrics['AUROC']:.4f}" if not np.isnan(metrics['AUROC']) else "N/A"
    })

wine_df = pd.DataFrame(wine_table_data)
print("="*100)
print("WINE DATASET RESULTS (5-Fold Cross-Validation)")
print("="*100)
display(wine_df)

## 9. Comparison Analysis

### Custom vs Scikit-learn Implementation Comparison

In [ ]:
all_results = {'Iris': iris_results, 'Wine': wine_results}

for dataset_name, results in all_results.items():
    print(f"\n{'='*70}")
    print(f"{dataset_name.upper()} DATASET ANALYSIS")
    print(f"{'='*70}")
    
    print("\n1. Accuracy Comparison (Custom vs Sklearn):")
    print("-" * 60)
    for algo in ['Decision Tree', 'Random Forest', 'Extra Trees']:
        custom_acc = results[f'Custom {algo}']['Accuracy']
        sklearn_acc = results[f'Sklearn {algo}']['Accuracy']
        diff = (custom_acc - sklearn_acc) * 100
        print(f"   {algo:15s}: Custom={custom_acc:.4f}, Sklearn={sklearn_acc:.4f} (Δ={diff:+.2f}%)")
    
    print("\n2. F1-Macro Comparison (Custom vs Sklearn):")
    print("-" * 60)
    for algo in ['Decision Tree', 'Random Forest', 'Extra Trees']:
        custom_f1 = results[f'Custom {algo}']['F1-Macro']
        sklearn_f1 = results[f'Sklearn {algo}']['F1-Macro']
        diff = (custom_f1 - sklearn_f1) * 100
        print(f"   {algo:15s}: Custom={custom_f1:.4f}, Sklearn={sklearn_f1:.4f} (Δ={diff:+.2f}%)")
    
    print("\n3. AUROC Comparison (Custom vs Sklearn):")
    print("-" * 60)
    for algo in ['Decision Tree', 'Random Forest', 'Extra Trees']:
        custom_auc = results[f'Custom {algo}']['AUROC']
        sklearn_auc = results[f'Sklearn {algo}']['AUROC']
        diff = (custom_auc - sklearn_auc) * 100
        print(f"   {algo:15s}: Custom={custom_auc:.4f}, Sklearn={sklearn_auc:.4f} (Δ={diff:+.2f}%)")
    
    # Ensemble vs single tree analysis
    custom_dt_acc = results['Custom Decision Tree']['Accuracy']
    custom_rf_acc = results['Custom Random Forest']['Accuracy']
    custom_et_acc = results['Custom Extra Trees']['Accuracy']
    
    print("\n4. Ensemble Methods Improvement over Decision Tree:")
    print("-" * 60)
    print(f"   Random Forest: Acc={custom_rf_acc:.4f} ({(custom_rf_acc - custom_dt_acc)*100:+.2f}% vs DT)")
    print(f"   Extra Trees:   Acc={custom_et_acc:.4f} ({(custom_et_acc - custom_dt_acc)*100:+.2f}% vs DT)")

## 10. Key Observations & Conclusions

In [ ]:
print("="*80)
print("KEY OBSERVATIONS")
print("="*80)
print("""
1. CUSTOM VS SKLEARN PERFORMANCE:
   - Custom implementations achieve comparable performance to scikit-learn.
   - Small differences are due to implementation details (tie-breaking, etc.)

2. ENSEMBLE METHODS IMPROVEMENT:
   - Random Forest and Extra Trees consistently outperform single Decision Trees.
   - This demonstrates the reduction in variance through ensemble averaging.

3. RANDOM FOREST VS EXTRA TREES:
   - Random Forest: Uses bootstrap sampling + optimal split selection
   - Extra Trees: Uses full dataset + random threshold selection
   - Both provide similar performance, but Extra Trees trains faster

4. BIAS-VARIANCE TRADEOFF:
   - Single Decision Trees have high variance (overfitting tendency)
   - Ensemble methods reduce variance by averaging multiple trees
   - Each tree sees different data (RF) or uses different splits (ET)

5. METRICS INTERPRETATION:
   - Accuracy: Overall correctness
   - F1-Macro: Balanced performance across all classes
   - AUROC: Ranking ability of the classifier
""")
print("="*80)
print("EXPERIMENT COMPLETE!")
print("="*80)